# Lesson 7: Case Study - Building a Complete ML Pipeline

## Learning Objectives

By the end of this lesson, you will be able to:
- Synthesize all learned OOP concepts (Classes, Inheritance, Polymorphism, Composition) to build a functional project.
- Design and implement a configurable, end-to-end machine learning pipeline from scratch.
- Use abstract base classes (ABCs) to define a common interface for pipeline components.
- Appreciate how an OOP design leads to clean, reusable, and extensible code.

## Why This Topic Matters

This lesson is where everything comes together. We've learned the individual notes and chords of OOP; now it's time to play a full song. Building a complete project from scratch solidifies understanding in a way that isolated examples cannot. 

We will build a flexible machine learning pipeline that is more robust and extensible than the one from the previous lesson. This case study will demonstrate how a thoughtful OOP design can produce professional-grade code that is a pleasure to work with, easy to modify, and simple to understand—the ultimate goal for any data scientist working in a team or on a long-term project.

## The Goal: A Configurable Prediction Pipeline

Our goal is to build a pipeline for a classification task using the well-known Iris dataset. The pipeline must be flexible enough to allow us to easily swap out its components (e.g., use a different model or a different data scaler) without rewriting the main logic.

The pipeline will have three main stages:
1.  **Load Data**: Load the Iris dataset.
2.  **Preprocess Data**: Scale the numerical features.
3.  **Train & Predict**: Train a model and use it to make predictions on new data.

## Step 1: Defining a Common Interface with Abstract Base Classes

In our last lesson, our pipeline worked because of 'duck typing'—as long as an object had a `.load()` or `.process()` method, the pipeline could use it. This is good, but we can be more explicit and enforce a contract.

An **Abstract Base Class (ABC)** defines a set of methods and properties that a subclass *must* implement. It creates a formal interface. Any class that inherits from the ABC must provide its own implementation of the abstract methods.

We'll define a `PipelineStep` ABC that has a single abstract method: `process()`.

In [ ]:
from abc import ABC, abstractmethod

class PipelineStep(ABC):
    """An abstract base class for all pipeline components."""
    
    @abstractmethod
    def process(self, data):
        """
        All pipeline steps must implement a process method.
        It takes some data and returns the processed data.
        """
        pass

# Try to instantiate it - this will fail!
try:
    step = PipelineStep()
except TypeError as e:
    print(f"Success! Error caught: {e}")

This ABC ensures that any component we build for our pipeline will have a consistent `process` method, making our system more predictable and robust.

## Step 2: Implementing the Concrete Components

Now, let's create our 'specialist' classes. Each one will inherit from `PipelineStep` and provide a concrete implementation of the `process` method. This is **Inheritance** and **Polymorphism** in action.

### Component A: The Data Loader

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

class IrisDataLoader(PipelineStep):
    """Loads the Iris dataset."""
    def process(self, data=None):
        """
        Loads the Iris dataset. Ignores input data.
        Returns a pandas DataFrame.
        """
        print("Step 1: Loading Iris Data...")
        iris = load_iris()
        df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
        df['target'] = iris.target
        return df

### Component B: The Preprocessors (Scalers)

Here we'll see the power of polymorphism. We will create two different scaler classes. Both are valid `PipelineStep`s and can be used interchangeably in our pipeline.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

class FeatureScaler(PipelineStep):
    """A base class for feature scalers."""
    def __init__(self):
        self.scaler = None
        self.trained = False
        
    def process(self, data):
        print(f"Step 2: Scaling features with {self.scaler.__class__.__name__}...")
        df = data.copy()
        feature_cols = [col for col in df.columns if col != 'target']
        
        if not self.trained:
            df[feature_cols] = self.scaler.fit_transform(df[feature_cols])
            self.trained = True
        else:
            df[feature_cols] = self.scaler.transform(df[feature_cols])
            
        return df

class StandardScalerStep(FeatureScaler):
    """A pipeline step that uses StandardScaler."""
    def __init__(self):
        super().__init__()
        self.scaler = StandardScaler()

class MinMaxScalerStep(FeatureScaler):
    """A pipeline step that uses MinMaxScaler."""
    def __init__(self):
        super().__init__()
        self.scaler = MinMaxScaler()

### Component C: The Model Trainer & Predictor

This component will be responsible for both training a model and, later, using it for prediction. It needs to handle two scenarios, so its `process` method will be a bit more complex.

In [ ]:
from sklearn.model_selection import train_test_split

class ModelTrainer(PipelineStep):
    """Trains a classifier and can also predict."""
    def __init__(self, model):
        self.model = model
        self.trained = False
        
    def process(self, data):
        df = data.copy()
        
        if not self.trained:
            # Training mode
            print(f"Step 3: Training {self.model.__class__.__name__}...")
            X = df.drop('target', axis=1)
            y = df['target']
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            self.model.fit(X_train, y_train)
            self.trained = True
            
            # During training, we can return the score
            score = self.model.score(X_test, y_test)
            print(f"Model accuracy: {score:.4f}")
            return score
        else:
            # Prediction mode
            print(f"Step 3: Predicting with {self.model.__class__.__name__}...")
            predictions = self.model.predict(df)
            return predictions

## Step 3: The Pipeline (The Conductor)

Now we build the main `Pipeline` class. It will be initialized with a list of `PipelineStep` objects. This is **Composition**. The `Pipeline` *has* a list of steps. Its `run` method will then iterate through these steps, passing the output of one step as the input to the next. This is a beautiful, clean, and scalable design.

In [ ]:
class Pipeline:
    """Manages a sequence of pipeline steps."""
    def __init__(self, steps):
        # Verify that all steps are valid PipelineStep instances
        if not all(isinstance(step, PipelineStep) for step in steps):
            raise TypeError("All steps must be instances of PipelineStep")
        self.steps = steps
        
    def run(self, initial_data=None):
        """Executes all steps in the pipeline in order."""
        data = initial_data
        for step in self.steps:
            data = step.process(data)
        return data

## Step 4: Assembly and Execution

This is the final, satisfying step where we assemble our pipeline and run it. Notice how readable this is. The configuration is separate from the execution, and we can easily see the sequence of operations.

### Scenario 1: Training with `StandardScaler` and `LogisticRegression`

In [ ]:
from sklearn.linear_model import LogisticRegression

print("--- SCENARIO 1: Training with Logistic Regression ---")

# 1. Configure and assemble the pipeline steps
training_steps = [
    IrisDataLoader(),
    StandardScalerStep(),
    ModelTrainer(model=LogisticRegression(random_state=42))
]

# 2. Create the pipeline
training_pipeline = Pipeline(steps=training_steps)

# 3. Run it
final_accuracy = training_pipeline.run()
print(f"Pipeline finished with final accuracy: {final_accuracy:.4f}")

### Scenario 2: Training with `MinMaxScaler` and `RandomForestClassifier`

Now for the magic. How much code do we need to change to use a different scaler and a different model? Almost none! We just assemble a new list of steps. This demonstrates the incredible flexibility of our design.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("\n--- SCENARIO 2: Training with Random Forest ---")

# 1. Assemble a DIFFERENT set of pipeline steps
rf_training_steps = [
    IrisDataLoader(),
    MinMaxScalerStep(), # Swapped the scaler
    ModelTrainer(model=RandomForestClassifier(random_state=42)) # Swapped the model
]

# 2. Create the new pipeline
rf_training_pipeline = Pipeline(steps=rf_training_steps)

# 3. Run it
rf_accuracy = rf_training_pipeline.run()

### Scenario 3: Running a Trained Pipeline for Prediction

What if we want to use a trained pipeline to make predictions on new data? Our design supports this too! We can create a *prediction* pipeline that reuses the *trained* scaler and model from our first scenario.

The `training_pipeline` object now holds the trained components. We can extract them to build a new pipeline for inference.

In [ ]:
print("\n--- SCENARIO 3: Predicting with the trained Logistic Regression pipeline ---")

# Create some new, unseen data
new_data = pd.DataFrame([
    [5.1, 3.5, 1.4, 0.2], # Expect class 0
    [6.7, 3.0, 5.2, 2.3], # Expect class 2
    [5.9, 3.0, 4.2, 1.5]  # Expect class 1
], columns=['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'])

# 1. Get the trained components from the first pipeline
trained_scaler = training_pipeline.steps[1]
trained_model_step = training_pipeline.steps[2]

# 2. Assemble a prediction pipeline
# Notice there is no data loader. We pass the data in directly.
prediction_pipeline = Pipeline(steps=[
    trained_scaler, 
    trained_model_step
])

# 3. Run the prediction
predictions = prediction_pipeline.run(initial_data=new_data)

print(f"\nPredictions for new data: {predictions}")

## Recap: The Power of OOP

Let's review how all the OOP concepts came together:

- **Classes:** We encapsulated logic and data into well-defined objects (`IrisDataLoader`, `StandardScalerStep`, `ModelTrainer`, `Pipeline`).
- **Abstraction (ABCs):** We defined a common interface (`PipelineStep`) that hides implementation details and enforces a contract for all our components.
- **Inheritance:** Our concrete steps inherited from the `PipelineStep` ABC, and our specific scalers inherited from a common `FeatureScaler` base class.
- **Polymorphism:** The `Pipeline` could work with `StandardScalerStep` or `MinMaxScalerStep` interchangeably because they both fulfilled the `PipelineStep` contract. The pipeline doesn't care about the specific type, only the interface.
- **Composition:** Our main `Pipeline` was built *from* `PipelineStep` objects. It *has* a list of steps. This allowed us to assemble and reassemble workflows with incredible flexibility.

This case study demonstrates that OOP is not just an academic exercise; it is a practical, powerful paradigm for building professional, maintainable, and scalable data science applications.

## Suggested Next Lesson

You have completed the core curriculum! The final step is to apply everything you've learned to a final project. You will be given a challenge: to take a messy, procedural data analysis script and refactor it into a clean, object-oriented pipeline, just like we did in this case study. This will be your opportunity to demonstrate your mastery of OOP for data science.